In [ ]:
#@title [준비] 실행 버튼(>)만 눌러주세요
import numpy as np, pandas as pd, matplotlib.pyplot as plt, matplotlib.font_manager as fm
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from scipy.optimize import minimize

# 한글 폰트 설정 (Colab 환경)
for font in fm.findSystemFonts(fontpaths=None, fontext='ttf'):
    if 'Nanum' in font:
        fm.fontManager.addfont(font)
plt.rcParams['font.family'] = 'NanumBarunGothic'
plt.rcParams['axes.unicode_minus'] = False

def _gen(seed, x_cfg, y_funcs, n=1000):
    rng = np.random.default_rng(seed)
    xs = {nm: rng.uniform(lo, hi, n) for nm, (lo, hi) in x_cfg.items()}
    ys = {nm: fn(xs, rng) for nm, fn in y_funcs.items()}
    return pd.DataFrame({**xs, **ys})

DATASETS = {
    '화학 공정': dict(seed=42,
        x_cfg={'온도_C': (50,200), '압력_kPa': (100,500), '유량_Lmin': (1,10), '교반_RPM': (100,800), '시간_min': (10,120)},
        y_funcs={
            '순도_pct': lambda x, r: np.clip(80+0.05*x['온도_C']+0.01*x['압력_kPa']-0.3*x['유량_Lmin']+0.005*x['교반_RPM']+0.02*x['시간_min']-0.0001*x['온도_C']*x['압력_kPa']+0.0002*x['온도_C']*x['교반_RPM']+r.normal(0,1.5,1000), 85, 99.9),
            '입자크기_um': lambda x, r: np.clip(25-0.05*x['온도_C']+0.01*x['압력_kPa']+0.8*x['유량_Lmin']-0.01*x['교반_RPM']-0.03*x['시간_min']+0.0005*x['유량_Lmin']*x['교반_RPM']+r.normal(0,1.0,1000), 1, 50),
            '점도_cP': lambda x, r: np.clip(50+0.3*x['온도_C']+0.1*x['압력_kPa']-2.0*x['유량_Lmin']+0.05*x['교반_RPM']+0.5*x['시간_min']-0.001*x['온도_C']*x['유량_Lmin']+r.normal(0,5,1000), 30, 400),
        }),
    '사출 성형': dict(seed=43,
        x_cfg={'사출압_bar': (50,200), '보압_bar': (30,120), '금형온도_C': (30,100), '냉각시간_sec': (5,60), '수지온도_C': (180,280)},
        y_funcs={
            '치수정밀도_mm': lambda x, r: np.clip(0.5-0.002*x['사출압_bar']+0.001*x['보압_bar']-0.003*x['금형온도_C']+0.005*x['냉각시간_sec']+0.001*x['수지온도_C']-0.00002*x['사출압_bar']*x['금형온도_C']+r.normal(0,0.02,1000), 0.01, 1.0),
            '수축률_pct': lambda x, r: np.clip(3.0-0.005*x['사출압_bar']-0.01*x['보압_bar']+0.02*x['금형온도_C']-0.01*x['냉각시간_sec']+0.005*x['수지온도_C']+r.normal(0,0.15,1000), 0.1, 5.0),
            '외관등급': lambda x, r: np.clip(8.0+0.005*x['사출압_bar']+0.01*x['보압_bar']-0.02*x['금형온도_C']+0.01*x['냉각시간_sec']-0.005*x['수지온도_C']+0.0001*x['사출압_bar']*x['보압_bar']+r.normal(0,0.3,1000), 1, 10),
        }),
    'CNC 정밀가공': dict(seed=44,
        x_cfg={'절삭속도_mpm': (50,300), '이송률_mmrev': (0.05,0.5), '절삭깊이_mm': (0.1,5.0), '공구마모_pct': (0,80), '냉각유량_Lmin': (1,10)},
        y_funcs={
            '표면조도_Ra': lambda x, r: np.clip(2.0-0.003*x['절삭속도_mpm']+3.0*x['이송률_mmrev']+0.2*x['절삭깊이_mm']+0.01*x['공구마모_pct']-0.05*x['냉각유량_Lmin']+0.05*x['이송률_mmrev']*x['공구마모_pct']+r.normal(0,0.1,1000), 0.1, 6.0),
            '치수편차_um': lambda x, r: np.clip(10-0.01*x['절삭속도_mpm']+5.0*x['이송률_mmrev']+1.5*x['절삭깊이_mm']+0.08*x['공구마모_pct']-0.3*x['냉각유량_Lmin']+r.normal(0,1.0,1000), 0.5, 30),
            '가공시간_sec': lambda x, r: np.clip(120-0.2*x['절삭속도_mpm']-50*x['이송률_mmrev']+15*x['절삭깊이_mm']+0.1*x['공구마모_pct']+0.5*x['냉각유량_Lmin']+r.normal(0,3,1000), 10, 300),
        }),
    '식품/바이오': dict(seed=45,
        x_cfg={'배양온도_C': (25,42), 'pH': (4.0,8.0), '배양시간_hr': (12,72), '교반속도_RPM': (50,300), '원료농도_pct': (1,10)},
        y_funcs={
            '생균수_logCFU': lambda x, r: np.clip(6.0+0.1*x['배양온도_C']-0.3*np.abs(x['pH']-6.5)+0.03*x['배양시간_hr']+0.002*x['교반속도_RPM']+0.05*x['원료농도_pct']-0.001*x['배양온도_C']*np.abs(x['pH']-6.5)+r.normal(0,0.2,1000), 5, 11),
            '안정성_month': lambda x, r: np.clip(18-0.2*x['배양온도_C']+0.5*(7-np.abs(x['pH']-6.5))-0.05*x['배양시간_hr']+0.01*x['교반속도_RPM']-0.1*x['원료농도_pct']+r.normal(0,1.0,1000), 1, 36),
            '수율_pct': lambda x, r: np.clip(40+0.5*x['배양온도_C']-2*np.abs(x['pH']-6.5)+0.3*x['배양시간_hr']+0.02*x['교반속도_RPM']+1.5*x['원료농도_pct']-0.01*x['배양시간_hr']*x['원료농도_pct']+r.normal(0,2,1000), 10, 95),
        }),
    '범용 제조': dict(seed=46,
        x_cfg={'설비온도_C': (30,250), '가동속도_rpm': (100,1500), '원료투입_kg': (10,100), '압력_bar': (1,20), '작업시간_min': (5,120)},
        y_funcs={
            '불량률_pct': lambda x, r: np.clip(8-0.01*x['설비온도_C']-0.001*x['가동속도_rpm']+0.02*x['원료투입_kg']-0.1*x['압력_bar']+0.01*x['작업시간_min']+0.00005*x['설비온도_C']*x['가동속도_rpm']+r.normal(0,0.5,1000), 0.1, 15),
            '수율_pct': lambda x, r: np.clip(70+0.03*x['설비온도_C']+0.005*x['가동속도_rpm']-0.05*x['원료투입_kg']+0.2*x['압력_bar']-0.02*x['작업시간_min']+r.normal(0,2,1000), 50, 99),
            '에너지효율_pct': lambda x, r: np.clip(85-0.02*x['설비온도_C']-0.005*x['가동속도_rpm']+0.01*x['원료투입_kg']+0.1*x['압력_bar']-0.05*x['작업시간_min']+r.normal(0,1.5,1000), 40, 99),
        }),
}

def load_dataset(name):
    global df, input_cols, output_cols, factory_model
    cfg = DATASETS[name]
    df = _gen(cfg['seed'], cfg['x_cfg'], cfg['y_funcs'])
    input_cols = list(cfg['x_cfg'].keys())
    output_cols = list(cfg['y_funcs'].keys())
    X_tr, X_te, y_tr, y_te = train_test_split(df[input_cols], df[output_cols], test_size=0.2, random_state=42)
    factory_model = RandomForestRegressor(n_estimators=100, random_state=42)
    factory_model.fit(X_tr, y_tr)
    y_pred = factory_model.predict(X_te)
    r2 = {col: r2_score(y_te.iloc[:,i], y_pred[:,i]) for i, col in enumerate(output_cols)}
    print(f'{name} 로드 완료 ({len(df)} 로트)')
    for col, score in r2.items():
        print(f'  {col:14s} R2={score:.3f}')

load_dataset('화학 공정')

print()
print('업종:')
for name in DATASETS:
    print(f'  load_dataset("{name}")')
print()
print('# Colab AI에게 시켜보세요:')
print('# "df 처음 10줄 보여줘"')
print('# "각 변수의 히스토그램을 그려줘"')
print('# "factory_model로 예측해줘"')
print('# "load_dataset으로 사출 성형 데이터로 바꿔줘"')